# Synthetic data generator #
The goal of this program is to generate synthetic data (ER,CM) with the same number of nodes and average degree of corresponding data networks.

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import networkx as nx
import matplotlib.pyplot as plt

 ## Real data ##

In [2]:
A = np.load("epidemic.npy")
G = nx.from_numpy_array(A)
N = G.number_of_nodes()
n_edg = G.number_of_edges()
orient = G.is_directed()
k = A.sum()/N

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Directed?" , G.is_directed())
print("Average degree:" , k)

Nodes: 95
Edges: 2360
Directed? False
Average degree: 49.68421052631579


## Synthetic data generation ##

In [3]:
import networkx as nx
import numpy as np
from tqdm import tqdm
import graph_tool.all as gt


N = A.shape[0]
k = np.mean(A.sum(axis=1))
p_er = k / (N - 1)

directed = G.is_directed()
z = 100
swap_factor = 20

# -------------------------------------------------
# Helper: A -> graph-tool Graph ( no self-loop)
# -------------------------------------------------
def A_to_graphtool(A, directed):
    A = np.asarray(A)
    src, dst = np.nonzero(np.triu(A, k=1))
    mask = src != dst  
    edges = np.column_stack([src[mask], dst[mask]])

    g = gt.Graph(directed=directed)
    g.add_vertex(A.shape[0])
    if edges.size > 0:
        g.add_edge_list(edges)
    return g

g0 = A_to_graphtool(A, directed=directed)


E0 = g0.num_edges()
n_iter = max(1, int(swap_factor * E0))

# -------------------------------------------------
# Container
# -------------------------------------------------
seeds = np.arange(z) + 100  # avoid low-numbered seeds
A_er_list = np.zeros((z, N, N))
A_cm_list = np.zeros((z, N, N))

# -------------------------------------------------
# Loop ensemble
# -------------------------------------------------
for i, seed in tqdm(
        enumerate(seeds),
        total=len(seeds),
        desc="Generating ER & CM ensembles"
    ):

    # ===== ER (NetworkX) =====
    G_er = nx.erdos_renyi_graph(
        N,
        p_er,
        seed=int(seed),
        directed=directed
    )
    A_er_list[i] = nx.to_numpy_array(G_er)

    # ===== CM (graph-tool) =====
    g = gt.Graph(g0)  

    gt.seed_rng(int(seed))   

    gt.random_rewire(
        g,
        model="configuration",
        n_iter=n_iter,
        edge_sweep=False
    )

    A_cm_list[i] = gt.adjacency(g).toarray()


np.savez_compressed(
    "ER_CM_ensemble.npz",
    seeds=seeds,
    A_er=A_er_list,
    A_cm=A_cm_list,
    p_er=p_er,
    swap_factor=swap_factor
)


Generating ER & CM ensembles: 100%|██████████| 100/100 [00:00<00:00, 115.34it/s]
